In [1]:
!pip -q install pyautogen groq

Question 5: Fact Checker Agent Flow (Truth vs. Fake Detection)

Task: Create a pipeline to verify a social media claim using 3 agents:
ClaimExtractorAgent: Extracts main claim from user input
FactCheckerAgent: Determines if it's real or fake
ReporterAgent: Outputs a news-style response


In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [3]:
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "base_url": "https://api.groq.com/openai/v1",
            "api_key": os.environ["GROQ_API_KEY"],
        }
    ],
    "temperature": 0.2,
}

Create the Agents

ClaimExtractorAgent

In [4]:
!pip install pyautogen==0.2.35

In [7]:
import os
from google.colab import userdata
from autogen import ConversableAgent

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


ClaimExtractorAgent

In [8]:
claim_extractor = ConversableAgent(
    name="ClaimExtractorAgent",
    system_message="""
You are a Claim Extraction Agent.

Responsibilities:
- Read the user's social media post.
- Identify the primary factual claim.
- Ignore opinions or emotional language.
- Return only the extracted claim.
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

FactCheckerAgent

In [9]:
fact_checker = ConversableAgent(
    name="FactCheckerAgent",
    system_message="""
You are a Fact Checker.

Responsibilities:
- Analyze the extracted claim.
- Decide whether it is:
  - True
  - False
  - Misleading
  - Unverified
- Explain your reasoning briefly.
- If evidence is insufficient, clearly state that the claim cannot be verified.
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

ReporterAgent

In [10]:
reporter = ConversableAgent(
    name="ReporterAgent",
    system_message="""
You are a News Reporter.

Responsibilities:
- Read the fact-checking result.
- Produce a short news-style report.

Format:

Headline

Summary

Verdict

Reason
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

User Input

In [11]:
claim = """
A social media post says that drinking hot water every hour completely cures diabetes within one week.
"""

Claim Extraction

In [12]:
claim_reply = claim_extractor.generate_reply(
    messages=[
        {
            "role": "user",
            "content": claim
        }
    ]
)

print("="*70)
print("CLAIM EXTRACTOR AGENT")
print("="*70)
print(claim_reply)

[autogen.oai.client: 08-05 08:29:58] {329} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.


CLAIM EXTRACTOR AGENT
Drinking hot water every hour cures diabetes within one week.


Fact Checking

In [13]:
fact_reply = fact_checker.generate_reply(
    messages=[
        {
            "role": "user",
            "content": claim_reply
        }
    ]
)

print("="*70)
print("FACT CHECKER AGENT")
print("="*70)
print(fact_reply)

[autogen.oai.client: 08-05 08:30:43] {329} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.


FACT CHECKER AGENT
**False**

This claim is false because there is no scientific evidence to support the idea that drinking hot water every hour can cure diabetes within one week. Diabetes is a complex medical condition that requires evidence-based treatment, such as medication, diet, and lifestyle changes, under the guidance of a healthcare professional. The American Diabetes Association and other reputable health organizations do not recommend drinking hot water as a cure for diabetes. In fact, attempting to treat diabetes with unproven remedies can be harmful and even life-threatening.


Reporter Agent

In [14]:
report_reply = reporter.generate_reply(
    messages=[
        {
            "role": "user",
            "content": fact_reply
        }
    ]
)

print("="*70)
print("REPORTER AGENT")
print("="*70)
print(report_reply)

[autogen.oai.client: 08-05 08:31:15] {329} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.


REPORTER AGENT
**Debunked: Hot Water Cure for Diabetes**

A recent claim that drinking hot water every hour can cure diabetes within a week has been proven to be false. Despite its simplicity, this remedy has no scientific basis and is not supported by reputable health organizations. In fact, attempting to treat diabetes with unproven methods can be detrimental to one's health.

**Verdict**
The claim that drinking hot water can cure diabetes is false.

**Reason**
There is no scientific evidence to support this claim, and reputable health organizations such as the American Diabetes Association do not recommend it as a treatment for diabetes. Instead, evidence-based treatments such as medication, diet, and lifestyle changes under the guidance of a healthcare professional are recommended to manage the condition.


Final Output

In [15]:
print("\n")
print("="*80)
print("FACT CHECKING REPORT")
print("="*80)

print("\nOriginal Social Media Claim")
print(claim)

print("\n")
print("="*80)
print("EXTRACTED CLAIM")
print("="*80)
print(claim_reply)

print("\n")
print("="*80)
print("FACT CHECK RESULT")
print("="*80)
print(fact_reply)

print("\n")
print("="*80)
print("NEWS REPORT")
print("="*80)
print(report_reply)



FACT CHECKING REPORT

Original Social Media Claim

A social media post says that drinking hot water every hour completely cures diabetes within one week.



EXTRACTED CLAIM
Drinking hot water every hour cures diabetes within one week.


FACT CHECK RESULT
**False**

This claim is false because there is no scientific evidence to support the idea that drinking hot water every hour can cure diabetes within one week. Diabetes is a complex medical condition that requires evidence-based treatment, such as medication, diet, and lifestyle changes, under the guidance of a healthcare professional. The American Diabetes Association and other reputable health organizations do not recommend drinking hot water as a cure for diabetes. In fact, attempting to treat diabetes with unproven remedies can be harmful and even life-threatening.


NEWS REPORT
**Debunked: Hot Water Cure for Diabetes**

A recent claim that drinking hot water every hour can cure diabetes within a week has been proven to be false

              Workflow
              --------
  
  
  
               Social Media Post
                      │
                      ▼
           ClaimExtractorAgent
          (Extract Main Claim)
                      │
                      ▼
            FactCheckerAgent
     (True / False / Misleading /
            Unverified)
                      │
                      ▼
             ReporterAgent
        (News-style Report)
                      │
                      ▼
          Final Fact Check Report

| Agent                   | Role                                                                                                                                 | Output            |
| ----------------------- | ------------------------------------------------------------------------------------------------------------------------------------ | ----------------- |
| **ClaimExtractorAgent** | Reads the social media post and extracts the main factual claim.                                                                     | Main claim        |
| **FactCheckerAgent**    | Evaluates the extracted claim and classifies it as **True**, **False**, **Misleading**, or **Unverified**, with a brief explanation. | Fact-check result |
| **ReporterAgent**       | Converts the fact-check result into a concise news-style report with a headline, summary, verdict, and reason.                       | News report       |


Data Flow
---------

| Step | Agent               | Input             | Output                        |
| ---- | ------------------- | ----------------- | ----------------------------- |
| 1    | ClaimExtractorAgent | Social media post | Extracted claim               |
| 2    | FactCheckerAgent    | Extracted claim   | Fact-check result             |
| 3    | ReporterAgent       | Fact-check result | News-style report             |
| 4    | Final Output        | All results       | Complete fact-checking report |
